#### CREATE `CUSTOMERS_VW` FROM VOLUME
- CATALOG NAME: GIZMO
- SCHEMA NAME: BRONZE
- VIEW NAME: CUSTOMERS_VW

In [0]:
%python
customers_df = (spark.read.format('json').load('/Volumes/gizmo/landing/operational_data/customers/'))
# display(customers_df.limit(10))
print(f'Row Count: {customers_df.count()}')

In [0]:
CREATE OR REPLACE VIEW GIZMO.BRONZE.CUSTOMERS_VW
AS
SELECT
customer_id,
customer_name,
to_date(date_of_birth,'yyyy-MM-dd') as date_of_birth,
email,
to_date(member_since,'yyyy-MM-dd') as member_since,
telephone,
to_timestamp(created_timestamp,'yyyy-MM-dd HH:mm:ss') as created_timestamp,
_metadata.file_path as file_path,
_metadata.file_name as file_name,
current_timestamp() as load_timestamp
 FROM json.`/Volumes/gizmo/landing/operational_data/customers/`;

#### QUERY `CUSTOMERS_VW` TO VALIDATE THE DATA

In [0]:
SELECT * FROM GIZMO.BRONZE.CUSTOMERS_VW;

In [0]:
%python
customers_count_df = spark.sql('''SELECT * FROM GIZMO.BRONZE.CUSTOMERS_VW''');
print(f'Row Count: {customers_count_df.count()}')

#### BELOW COMMAND TO EXECUTE THE FUNCTION

In [0]:
%run /Workspace/Users/pde1409@hotmail.com/AzureDatabricks-Gizmo/AzureDatabricks-GizmoBox/01.GizmoBox/02.Config/01.config.py

In [0]:
%python
try:
    verify_pipeline_counts(customers_df, customers_count_df, "01.IngestCustomersJSON")
except AssertionError as e:
    # This ensures the notebook actually fails if scheduled via a Databricks Workflow/Job
    raise e

In [0]:
%skip
%python
spark.sql("""
CREATE OR REPLACE TABLE GIZMO.BRONZE.INGEST_LOGS (
  log_id STRING,
  event_time TIMESTAMP,
  event_type STRING,
  source_table STRING,
  target_table STRING,
  record_count BIGINT,
  status STRING,
  message STRING,
  user_name STRING,
  notebook_path STRING,
  pipeline_name STRING
)
COMMENT 'Logging table for data ingestion events and pipeline activity'
""")

#### CAPTURE AUDIT / OBSERVABILITY MECHANISM 

In [0]:
%skip
%python
from pyspark.sql import Row
from datetime import datetime
import uuid

load_start_time = datetime.now()
record_count = customers_count_df.count()
load_end_time = datetime.now()

user_name = 'pde1409'
log_entry = Row(
    log_id=str(uuid.uuid4()),
    event_time=current_date,
    event_type="LOAD",
    source_table="json.`/Volumes/gizmo/landing/operational_data/customers/`",
    target_table="GIZMO.BRONZE.CUSTOMERS_VW",
    record_count=record_count,
    status="SUCCESS",
    message="Loaded customers data into bronze view",
    user_name=user_name,
    notebook_path=dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get(),
    pipeline_name='01.IngestCustomersJSON',
    load_start_time=load_start_time,
    load_end_time=load_end_time
)

In [0]:
%python
from pyspark.sql import Row
from datetime import datetime
import uuid
import pyspark.sql.functions as F

# 1. Timestamps & Dates
if 'load_start_time' not in locals():
    load_start_time = datetime.now() 

load_end_time = datetime.now()
current_date = load_end_time.date() # YYYY-MM-DD

# ---------------------------------------------------------
# NEW LOGIC: Calculate Sequential Run ID for Today
# ---------------------------------------------------------
try:
    # Query the audit table for the max run_id logged today
    # Casting to INT during the max check ensures correct numerical sorting
    max_run_df = spark.table("GIZMO.BRONZE.AUDIT_LOGS") \
        .filter(F.col("event_time") == current_date) \
        .select(F.max(F.col("run_id").cast("int")).alias("max_id"))
    
    max_id_row = max_run_df.collect()[0]
    
    if max_id_row["max_id"] is not None:
        # If runs exist today, increment the max value by 1
        next_run_int = max_id_row["max_id"] + 1
    else:
        # If this is the first run of the day, start at 1
        next_run_int = 1
        
except Exception as e:
    # Fallback if the table is completely empty or hasn't been initialized yet
    next_run_int = 1

# Format the integer as a 2-digit string with leading zeros (e.g., 1 -> "01", 10 -> "10")
run_id_str = f"{next_run_int:02d}"
# ---------------------------------------------------------

# 2. Grab Databricks environment metadata safely
try:
    context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    user_name = context.tags().apply("user")
    pipeline_name = '01.IngestCustomersJSON' #context.tags().asJavaMap().get("jobName") or notebook_path.split("/")[-1]
except Exception:
    notebook_path = "dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()"
    user_name = "System"
    pipeline_name = pipeline_name

# 3. Extract your data count
record_count = customers_count_df.count()

# 4. Construct the row matching your specific DDL column types
log_entry = Row(
    log_id=str(uuid.uuid4()),
    run_id=run_id_str,                # Formatted sequential string ("01", "02")
    event_time=current_date,          
    event_type="LOAD",
    source_table="json.`/Volumes/gizmo/landing/operational_data/customers/`",
    target_table="GIZMO.BRONZE.CUSTOMERS_VW",
    record_count=record_count,
    status="SUCCESS",
    message="Loaded customers data into bronze view",
    user_name=user_name,
    notebook_path=notebook_path,
    pipeline_name=pipeline_name,
    load_start_time=load_start_time,  
    load_end_time=load_end_time       
)

# 5. Define the strict schema mapping (Notice StringType for run_id)
from pyspark.sql.types import StructType, StructField, StringType, DateType, LongType, TimestampType

log_schema = StructType([
    StructField("log_id", StringType(), True),
    StructField("run_id", StringType(), True),  # Changed to StringType to match DDL change
    StructField("event_time", DateType(), True),
    StructField("event_type", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("record_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
    StructField("user_name", StringType(), True),
    StructField("notebook_path", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("load_start_time", TimestampType(), True),
    StructField("load_end_time", TimestampType(), True)
])



In [0]:
%python
# 6. Convert to DataFrame and append to Delta
log_entry_df = spark.createDataFrame([log_entry], schema=log_schema)
log_entry_df.write.format("delta").mode("append").saveAsTable("GIZMO.BRONZE.AUDIT_LOGS")

In [0]:
%python
dbutils.notebook.exit("CUSTOMERS HAS BEEN LOADED SUCCESSFULLY INTO GIZMO.BRONZE.CUSTOMERS_VW")

In [0]:
SELECT * FROM GIZMO.BRONZE.AUDIT_LOGS;

In [0]:
%sql
DROP TABLE GIZMO.BRONZE.AUDIT_LOGS